# Generación del modelo

## Pasar los jsonl a datos en csv para poder cargarlos en la base de datos de la API

In [1]:
import pandas as pd
import duckdb
import json

In [3]:
# reviews_file = 'Kindle_Store.jsonl'
# reviews = pd.read_json(reviews_file, lines=True, chunksize=10)[['rating', 'title', 'parent_asin', 'user_id']]
# for chunk in reviews:
#     first_review = chunk.iloc[1]
#     print(first_review['rating'], first_review['title'], first_review['parent_asin'], first_review['user_id'])
#     break

In [ ]:
# reviews_file = 'meta_Kindle_Store.jsonl'
# books = pd.read_json(reviews_file, lines=True, chunksize=10)
# for chunk in books:
#     first_book = chunk.iloc[1]
#     # print(first_book)
#     # break
#     print(first_book['title'], first_book['subtitle'], first_book['parent_asin'], first_book['categories'], first_book['images'], first_book['description'], first_book['details'])
#     break

Microsoft PowerPoint 2016 2013 2010 2007 Tips Tricks and Shortcuts: Presentations, Special Effects and Animations in 25 Mini-Lessons [Print Replica] Kindle Edition B07DH1LF1K ['Kindle Store', 'Kindle eBooks', 'Computers & Technology'] [{'large': 'https://m.media-amazon.com/images/I/51DS1veLPrL.jpg', 'variant': 'MAIN'}] ['From the Author', 'Amelia Griggs is an Instructional Designer and eLearning developer. She has a M.S. in Instructional Design and Technology from Philadelphia University, and a B.B.A in Computer Science from Temple University. In addition to designing and developing all kinds of educational materials, including instructional videos, Amelia also loves writing educational articles, fictional short stories and poetry.\xa0After collecting all kinds of tips, tricks and shortcuts while conducting Microsoft Office classroom training and online training, Amelia created a Microsoft Office How-To Series consisting of 3 books:', 'Microsoft Word Tips, Tricks and Shortcuts: 35 Mini

In [2]:
con = duckdb.connect('amazon_etl.db')

### Obtener libros con más de 20 reseñas

In [21]:
con.execute("""
    DROP TABLE IF EXISTS popular_books;
    CREATE TEMP TABLE popular_books AS
    SELECT parent_asin
    FROM read_json_auto('Kindle_Store.jsonl')
    GROUP BY parent_asin
    HAVING COUNT(*) >= 500;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [22]:
total_books = con.execute("""
    SELECT COUNT(*) as total_books
    FROM popular_books
""").df()

total_books

,total_books
0,4601


### Encontrar usuarios con al menos 15 reseñas en total dentro de esos libros

In [43]:
con.execute("""
    DROP TABLE IF EXISTS active_users;
    CREATE TEMP TABLE active_users AS
    SELECT user_id
    FROM read_json_auto('Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books)
    GROUP BY user_id
    HAVING COUNT(*) BETWEEN 35 AND 50;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [44]:
total_users = con.execute("""
    SELECT COUNT(*) as total_users
    FROM active_users
""").df()

total_users

,total_users
0,2709


### Obtener sólo las reviews de los libros y usuarios filtrados

In [45]:
con.execute("""
    DROP TABLE IF EXISTS final_reviews;
    CREATE TEMP TABLE final_reviews AS
    SELECT user_id, parent_asin as item_id, rating
    FROM read_json_auto('Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books)
    AND user_id IN (SELECT user_id FROM active_users);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [46]:
total_reviews = con.execute("""
SELECT COUNT(*) as total_reviews
FROM final_reviews
""").df()

total_reviews

,total_reviews
0,111005


### Obtener metadatos de los libros seleccionados

In [47]:
con.execute("""
    CREATE TEMP TABLE final_meta AS
    SELECT DISTINCT parent_asin as item_id, title, subtitle, description, categories, images
    FROM read_json_auto('meta_Kindle_Store.jsonl')
    WHERE parent_asin IN (SELECT parent_asin FROM popular_books);
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [48]:
total_books = con.execute("""
SELECT COUNT(*) as total_books
FROM final_meta
""").df()

total_books

,total_books
0,4601


In [49]:
meta_sample = con.execute("""
    select * from final_meta
    limit 5;
""").df()

print(meta_sample.iloc[0])

item_id                                               B0030ZRZ7S
title          SILENT WITNESS: A Josie Bates Thriller (The Wi...
subtitle                                          Kindle Edition
description                                                   []
categories     [Kindle Store, Kindle eBooks, Literature & Fic...
images         [{'large': 'https://m.media-amazon.com/images/...
Name: 0, dtype: object


In [50]:
reviews_sample = con.execute("""
    select * from final_reviews
    limit 5;
""").df()

print(reviews_sample.iloc[0])

user_id    AGHFDAIJOJPZBFMCZSEAHQOSGOIQ
item_id                      B0BGXYL3BZ
rating                              4.0
Name: 0, dtype: object


### Exportar a CSV

In [51]:
con.execute("COPY final_reviews TO 'kindle_reviews_seed.csv' (HEADER, DELIMITER ',')")
con.execute("COPY final_meta TO 'kindle_meta_seed.csv' (HEADER, DELIMITER ',')")

In [52]:
meta_df = pd.read_csv('kindle_meta_seed.csv')

In [53]:
print(meta_df.iloc[0]['categories'])

[Kindle Store, Kindle eBooks, Literature & Fiction]


In [54]:
rows_with_unwanted_categories = meta_df[meta_df['categories'].str.contains('Kindle', na=False)]
rows_with_unwanted_categories.count()

item_id        4601
title          4601
subtitle       4214
description    4601
categories     4601
images         4601
dtype: int64

### Mantener sólo las categorías relevantes

In [55]:
unwanted_categories = ['Kindle Store', 'Kindle eBooks', 'Kindle Unlimited', 'Kindle Short Reads', 'Kindle eTextbooks', 'Kindle Games & Active Content']
for category in unwanted_categories:
    meta_df['categories'] = meta_df['categories'].apply(lambda x: x.replace(category + ',', '') if pd.notnull(x) else x)

meta_df['categories'] = meta_df['categories'].apply(lambda x: x.replace('Kindle eBooks', '') if pd.notnull(x) else x)

meta_df['categories'] = meta_df['categories'].apply(lambda x: x.strip() if pd.notnull(x) else x)

In [56]:
rows_with_unwanted_categories = meta_df[meta_df['categories'].str.contains('Kindle', na=False)]
rows_with_unwanted_categories.count()

item_id        0
title          0
subtitle       0
description    0
categories     0
images         0
dtype: int64

In [57]:
meta_df.head()

,item_id,title,subtitle,description,categories,images
0,B0030ZRZ7S,SILENT WITNESS: A Josie Bates Thriller (The Wi...,Kindle Edition,[],[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...
1,B00I8YB71Q,You Are Not Small,Kindle Edition,"[Amazon.com Review, A Look Inside, 'You Are (N...",[ 'Children\'s eBooks'],[{'large': 'https://m.media-amazon.com/images/...
2,B00F27B8YI,BLACK: (Humorous detective mystery),Kindle Edition,"[Review, 'NY Times & USA Today Bestselling Aut...",[ Literature & Fiction],[{'large': 'https://m.media-amazon.com/images/...
3,B00JCS2GTY,Skink--No Surrender (Skink Series Book 7),Kindle Edition,"[From School Library Journal, 'Gr 9 Up—Richard...",[ 'Children\'s eBooks'],[]
4,B0739Z4XJ9,The Other Woman: A Novel (Gabriel Allon Book 18),Kindle Edition,"[From the Inside Flap, 'From Daniel Silva, the...",[ Literature & Fiction],[]


### Exportar a csv

In [58]:
meta_df.to_csv('kindle_meta_cleaned.csv', index=False)
meta_df.head(100).to_csv('kindle_meta_cleaned_sample.csv', index=False)

In [59]:
len(meta_df)

4601

In [60]:
reviews_df = pd.read_csv('kindle_reviews_seed.csv')

In [61]:
reviews_df.head(100).to_csv('kindle_reviews_sample.csv', index=False)

In [62]:
reviews_df['user_id'].unique()

<StringArray>
[  'AGHFDAIJOJPZBFMCZSEAHQOSGOIQ',   'AEVXW3KVAVIE4FPVLOCACZREVO5A',
   'AHEBJ5HU5XFVKHIUA3FLGMOPHOLA',   'AFJLT4MSWKEXAG2KBDP26DLKBLCA',
   'AE2EI2FLRSDLXEJSWIOX5AO5T5YQ',   'AG2SN33FT6SSQIW36QB2GJITQ2DA',
   'AGS7OQZUP4EOJVK25NC72LEQ6YFA',   'AHC6DWAVLH4BQRE5H4KLCVXVEVZA',
   'AH5JAIGUCANQ7TZENRTLSO74QQXQ',   'AEZSSK6LVSCTAIWK3ESS7TR2J4XQ',
 ...
   'AECCEXRYXAFKLBRLXSLG434XB2LQ',   'AFDLHTCJS4GZSV4NS5JBTPE467FQ',
 'AHVERFH5M2H76YELYO53EMZNKB7Q_1',   'AF4OMXALFVBMFOFAYOUSFS4P33RQ',
   'AFWNAURAFBMNLH5M7OXISGBFYRNQ',   'AH2UW4UZ4OVDVEXEFAXGDRXKYOCQ',
   'AGS3L6QXRPQUCSPJUUMU6OB6LFEQ',   'AGDJBU7AENWOHFDWJQID5M2FARVQ',
   'AGVSC2JFHE3RFEI7JCNMB5EDKINQ',   'AEHC5F4PFXFOZ4Q63W3WYJBIKLJA']
Length: 2709, dtype: str

In [63]:
reviews_df.head()

,user_id,item_id,rating
0,AGHFDAIJOJPZBFMCZSEAHQOSGOIQ,B0BGXYL3BZ,4.0
1,AGHFDAIJOJPZBFMCZSEAHQOSGOIQ,B07Q5TL9SQ,3.0
2,AGHFDAIJOJPZBFMCZSEAHQOSGOIQ,B076CJVFKF,4.0
3,AGHFDAIJOJPZBFMCZSEAHQOSGOIQ,B0BBL2ZW73,5.0
4,AGHFDAIJOJPZBFMCZSEAHQOSGOIQ,B0B1XQTZ1R,4.0


In [64]:
len(reviews_df)

111005